# Giai đoạn 2 — Trích xuất embedding từ ảnh thật
Gắn source code v2, bộ dữ liệu giai đoạn 1 v2 và H&M competition. Bật GPU và Internet cho lần tải encoder đầu tiên. Nếu thiếu thư viện, cài `requirements-extraction.txt`; không cần cài lại mỗi lần chạy.

In [ ]:
from pathlib import Path
import os
import sys

# Set this only if more than one source-code dataset is attached.
CODE_ROOT_OVERRIDE = None
if CODE_ROOT_OVERRIDE:
    CODE_ROOT = Path(CODE_ROOT_OVERRIDE).resolve()
else:
    candidates = {p.resolve() for p in [Path.cwd(), Path.cwd().parent] if (p/'src/config.py').is_file()}
    if Path('/kaggle/input').exists():
        for root, dirs, files in os.walk('/kaggle/input'):
            dirs[:] = [d for d in dirs if d not in {'images', '.git', '__pycache__'}]
            candidate = Path(root)
            if candidate.name == 'src' and 'config.py' in files and (candidate/'pipeline/train_baselines.py').is_file():
                candidates.add(candidate.parent.resolve())
    if len(candidates) != 1:
        raise ValueError(f'Set CODE_ROOT_OVERRIDE to the folder containing src. Candidates: {sorted(candidates)}')
    CODE_ROOT = next(iter(candidates))
if not (CODE_ROOT/'src/config.py').is_file():
    raise FileNotFoundError('The selected code folder is not source v2')
sys.path.insert(0, str(CODE_ROOT))
from src.utils.paths import resolve_data_dir, resolve_raw_dir, resolve_embeddings_dir
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').exists() else CODE_ROOT
print('Code:', CODE_ROOT, 'Output:', WORK_ROOT)


In [ ]:
DATA_DIR_OVERRIDE = None
RAW_DATA_OVERRIDE = None
IMAGES_DIR_OVERRIDE = None
DATA_DIR = resolve_data_dir(CODE_ROOT, DATA_DIR_OVERRIDE)
RAW_DIR = resolve_raw_dir(CODE_ROOT, RAW_DATA_OVERRIDE)
IMAGES_DIR = Path(IMAGES_DIR_OVERRIDE) if IMAGES_DIR_OVERRIDE else RAW_DIR/'images'
OUTPUT_DIR = WORK_ROOT/'embeddings/v2'
SELECTED_PATH = DATA_DIR/'selected_articles.csv'
BATCH_SIZE = 64
MAX_MISSING_IMAGE_FRACTION = 0.01  # allow up to 1% natural image missingness; keep availability masks
IMAGE_REVISION = 'main'  # use the resolved commit in metadata to reproduce a later extraction
TEXT_REVISION = 'main'
import torch
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Selection:', SELECTED_PATH, 'Images:', IMAGES_DIR, 'Output:', OUTPUT_DIR)


## 1. Xem ảnh đầu vào
Đây phải là ảnh sản phẩm thật, không phải ảnh đen thay thế. Pipeline tiếp theo sẽ kiểm tra toàn bộ ảnh và xuất image_audit.json.

In [ ]:
import pandas as pd
from IPython.display import display
from PIL import Image, UnidentifiedImageError
from src.features.extract_embeddings import image_path
selected = pd.read_csv(SELECTED_PATH, dtype={'article_id': str})
for key in selected.article_id.sample(min(10, len(selected)), random_state=42):
    path = image_path(IMAGES_DIR, key)
    print(key, path)
    try:
        with Image.open(path) as source:
            preview = source.convert("RGB")
            preview.thumbnail((224, 224))
            display(preview)
    except (OSError, UnidentifiedImageError) as exc:
        print(f"Không đọc được ảnh mẫu: {key} ({type(exc).__name__}); audit toàn bộ ở cell tiếp theo.")


## 2. Trích xuất và kiểm định
Ảnh 768d và text 384d được L2-normalize riêng. Hai encoder chưa có không gian ngữ nghĩa chung. Không tái tạo text theo công thức khác: dùng đúng text_for_embedding của Stage 1.

Notebook cho phép tối đa 1% ảnh thiếu/hỏng có ghi nhận availability. Ảnh thiếu không đi qua encoder, chỉ lưu zero placeholder và mask. Xem missing_images.csv để phân biệt missing/unreadable. Đây là chế độ thiếu tự nhiên; pipeline Stage 3 hiện vẫn yêu cầu full modalities, vì vậy cần xử lý tập complete-case đồng bộ hoặc bổ sung chế độ đánh giá thiếu tự nhiên trước khi dùng bộ này để train. Không đổi hash hoặc availability để vượt kiểm định.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.features.extract_embeddings import extract_embeddings

# Cho phép tối đa 1% ảnh thiếu/hỏng; không thay ảnh lỗi bằng ảnh đen.
MAX_MISSING_IMAGE_FRACTION = 0.01
OUTPUT_DIR = Path(OUTPUT_DIR)

try:
    report = extract_embeddings(
        selected_articles_path=SELECTED_PATH,
        images_dir=IMAGES_DIR,
        output_dir=OUTPUT_DIR,
        batch_size=BATCH_SIZE,
        max_missing_image_fraction=MAX_MISSING_IMAGE_FRACTION,
        image_revision=IMAGE_REVISION,
        text_revision=TEXT_REVISION,
    )
except ValueError as exc:
    if str(exc).startswith("Image audit failed:"):
        audit = pd.read_csv(OUTPUT_DIR / "availability.csv", dtype={"article_id": str})
        display(audit.loc[audit["image_available"].eq(False)])
    raise

availability = pd.read_csv(OUTPUT_DIR / "availability.csv", dtype={"article_id": str})
missing = availability.loc[availability["image_available"].eq(False)]
missing.to_csv(OUTPUT_DIR / "missing_images.csv", index=False)
print(f"Ảnh thiếu/hỏng: {len(missing)}/{len(availability)} ({len(missing)/len(availability):.2%})")
print("Ảnh thiếu được lưu zero vector + image_available=False; đây chưa phải bộ full modalities.")
display(missing)
display(report)


## 3. Tải đúng kết quả vừa tạo
Archive chứa cả ảnh, text, availability, metadata, audit và validation report. Không copy từ dataset embedding cũ trong Input.

In [ ]:
import shutil
from IPython.display import FileLink
archive = shutil.make_archive(str(WORK_ROOT/'hm-embeddings-v2'), 'zip', root_dir=OUTPUT_DIR)
display(FileLink(archive))
